# Pipeline end-to-end: Preprocesado → CNN-AE → TCN → Semáforo

Este notebook orquesta todo el flujo del repo:
1) Preprocesado (como en `eda_notebook`)
2) Entrenamiento del `CNN-AE` para aprender la 'normalidad' y obtener el MSE por ventana
3) Calibrado de umbrales y visualización del semáforo a partir del MSE
4) Uso del `CNN-AE` como extractor congelado y entrenamiento de una `TCN` + cabezas multitarea para producir señales BUY/HOLD/SELL

Cada sección explica qué hace y por qué antes del código, y se basa en los módulos `load_data.py`, `cnn_ae.py` y el notebook TCN ya presente en el repo.

**Nota:** si ejecutas en Colab, descomentad las instalaciones en la siguiente celda. En local pueden no ser necesarias.

In [ ]:
# (Opcional Colab) Instalaciones recomendadas - ejecutar solo en entorno limpio
# !pip install -q numpy<2.3 pandas<3 numba yfinance pandas_ta torch torchvision scikit-learn matplotlib
# Reiniciar kernel si reinstalas numpy/pandas/torch en Colab

# Añadir repo al path (si corres desde otra carpeta)
import sys, os
repo_root = os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler

# Importar utilidades del repo
from load_data import DataProcessor
from cnn_ae import ConvAutoencoder, train_autoencoder, compute_reconstruction_errors, calibrate_thresholds, classify_signals, save_model, load_model, plot_training_history, plot_semaforo

print('Imports OK')

## 1) Preprocesado (EDA transformado a pasos reproducibles)
Explicación: usamos `DataProcessor` para descargar precios, añadir features técnicas y dividir en periodo de estabilidad (train) y periodo de test. Seleccionamos las `FEATURE_COLS` que usa el `CNN-AE`.

In [ ]:
# Parámetros
TICKER = '^IBEX'
START_DATE = '2000-01-01'
END_DATE = '2024-12-31'
STABILITY_END = '2019-12-31'
WINDOW_SIZE = 30
FEATURE_COLS = [
, 
, 
, 
, 
]

# 1. Descargar y calcular features
processor = DataProcessor(ticker=TICKER, start_date=START_DATE, end_date=END_DATE)
df = processor.download_data()
processor.add_features()
df = processor.data

# 2. Seleccionar las columnas que usa el modelo y dividir (temporal)
full_df = df[FEATURE_COLS].copy()
train_df, test_df = full_df[:STABILITY_END], full_df[STABILITY_END:]
print(f'Train rows: {len(train_df)} | Test rows: {len(test_df)}')

# 3. Escalado (fitar solo con train) y creación de ventanas deslizantes
scaler = MinMaxScaler().fit(train_df)
train_scaled = scaler.transform(train_df)
test_scaled  = scaler.transform(test_df)

def make_windows(arr, ws=WINDOW_SIZE):
    return np.array([arr[i:i+ws] for i in range(len(arr) - ws)])

X_train = make_windows(train_scaled, WINDOW_SIZE)
X_test  = make_windows(test_scaled, WINDOW_SIZE)

# Fechas alineadas con cada ventana (la ventana i termina en index i+ws-1)
train_dates = train_df.index[WINDOW_SIZE:]
test_dates  = test_df.index[WINDOW_SIZE:]

print('X_train shape =', X_train.shape, 'X_test shape =', X_test.shape)

## 2) Entrenamiento del `CNN-AE`
Explicación: el autoencoder convolucional se entrena SOLO con `X_train` (periodo de estabilidad). Tras el entrenamiento guardamos pesos y trazamos la pérdida de entrenamiento/validación para comprobar convergencia.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

n_features = len(FEATURE_COLS)
model = ConvAutoencoder(n_features=n_features, seq_len=WINDOW_SIZE)
history = train_autoencoder(model, X_train, epochs=50, batch_size=64, lr=1e-3, val_split=0.1, device=device)
plot_training_history(history)
save_model(model, path='cnn_ae_weights.pth')

## 3) Inferencia con CNN-AE: MSE por ventana y semáforo básico
Explicación: calculamos el error de reconstrucción (MSE) para cada ventana. Con los errores del set de entrenamiento calibramos percentiles para definir Verde/Ámbar/Rojo.

In [ ]:
# Errores en train (para calibrar)
train_errors = compute_reconstruction_errors(model, X_train, batch_size=256, device=device)
thresholds = calibrate_thresholds(train_errors, p_amber=90, p_red=97)

# Errores en test y semáforo simple por MSE
test_errors = compute_reconstruction_errors(model, X_test, batch_size=256, device=device)
plot_semaforo(test_dates, test_errors, thresholds, title='Semáforo MSE (CNN-AE)')

# Clasificación por MSE
labels = classify_signals(test_errors, thresholds)
from collections import Counter
print('Distribución semáforo MSE (test):', Counter(labels))

## 4) TCN: usar CNN-AE como extractor frozen y entrenar agente multitarea
Explicación: el encoder del `CNN-AE` genera vectores latentes `z` por ventana. Construimos secuencias de `z` (historia temporal) y las pasamos a una `TCN` que alimenta dos cabezas: clasificación (BUY/HOLD/SELL) y regresión (retorno esperado). El diseño sigue el notebook `tcn_agent_semaforoV3Bueno2.ipynb`.

In [ ]:
# --- Definiciones TCN y agente (extraído y adaptado del notebook TCN existente) ---
import torch.nn as nn
import torch.nn.functional as F

class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, dilation):
        super().__init__()
        self.pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation, padding=0)
    def forward(self, x):
        return self.conv(F.pad(x, (self.pad, 0)))

class ResidualTCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, dilation, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(in_ch, out_ch, kernel, dilation),
            nn.BatchNorm1d(out_ch), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(out_ch, out_ch, kernel, dilation),
            nn.BatchNorm1d(out_ch), nn.ReLU(), nn.Dropout(dropout),
        )
        self.downsample = (nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity())
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.net(x) + self.downsample(x))

class TCN(nn.Module):
    def __init__(self, input_dim, channels, kernel=3, dropout=0.1, output_dim=64):
        super().__init__()
        layers, in_ch = [], input_dim
        for i, out_ch in enumerate(channels):
            layers.append(ResidualTCNBlock(in_ch, out_ch, kernel, 2 ** i, dropout))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Linear(channels[-1], output_dim)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.tcn(x)[:, :, -1]
        return self.head(x)

class SemaforoAgent(nn.Module):
    def __init__(self, z_dim, tcn_channels=None, tcn_kernel=3, dropout=0.3, context_dim=64):
        super().__init__()
        if tcn_channels is None:
            tcn_channels = [32, 16]
        self.tcn = TCN(z_dim, tcn_channels, tcn_kernel, dropout, context_dim)
        self.cls = nn.Sequential(nn.Linear(context_dim, 64), nn.ReLU(), nn.Linear(64, 3))
        self.reg = nn.Sequential(nn.Linear(context_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, z):
        ctx = self.tcn(z)
        return self.cls(ctx), self.reg(ctx).squeeze(-1)

# Helpers: generar Z sequences usando el encoder congelado del CNN-AE
def build_zf_sequences(cnn_ae, X_windows, feat_windows, tcn_seq_len, device, batch_size=256):
    cnn_ae.eval()
    all_z = []
    for i in range(0, len(X_windows), batch_size):
        batch = torch.tensor(X_windows[i:i+batch_size], dtype=torch.float32).to(device)
        with torch.no_grad():
            all_z.append(cnn_ae.get_latent_flat(batch).cpu().numpy())
    all_z = np.concatenate(all_z, axis=0).astype(np.float32)
    all_zf = np.concatenate([all_z, feat_windows.astype(np.float32)], axis=1)
    return np.array([all_zf[i:i+tcn_seq_len] for i in range(len(all_zf) - tcn_seq_len)])

def forward_returns(close, decision_offset, n_seqs, horizon):
    c = close.values.astype(np.float64)
    out = []
    for i in range(n_seqs):
        d = decision_offset + i
        if d + horizon >= len(c):
            break
        out.append(np.log(c[d + horizon] / c[d]))
    return np.asarray(out, dtype=np.float32)

def make_labels(fwd, thr=0.0):
    y = np.zeros(len(fwd), dtype=np.int64)
    y[fwd > thr] = 1
    y[fwd < -thr] = 2
    return y

def train_supervised(agent, z_train, fwd_train, r1_train, epochs=200, lr=3e-4, weight_decay=1e-3, thr=0.0, val_frac=0.2, batch_size=128, patience=40, min_epochs=40, label_smoothing=0.05, lambda_reg=10.0, device='cpu'):
    y_all = make_labels(fwd_train, thr)
    n_val = int(len(z_train) * val_frac)
    Xtr = torch.as_tensor(z_train[:-n_val], dtype=torch.float32, device=device)
    ytr = torch.as_tensor(y_all[:-n_val],   dtype=torch.long,    device=device)
    rtr = torch.as_tensor(r1_train[:-n_val], dtype=torch.float32, device=device)
    Xva = torch.as_tensor(z_train[-n_val:], dtype=torch.float32, device=device)
    yva = torch.as_tensor(y_all[-n_val:],   dtype=torch.long,    device=device)
    rva = torch.as_tensor(r1_train[-n_val:], dtype=torch.float32, device=device)
    fwd_va = fwd_train[-n_val:]

    params = list(agent.tcn.parameters()) + list(agent.cls.parameters()) + list(agent.reg.parameters())
    opt = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    n = Xtr.shape[0]
    best_loss, best_state, best_ep, bad = float('inf'), None, 0, 0
    val_acc_hist = []
    for ep in range(epochs):
        agent.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            logits, rpred = agent(Xtr[idx])
            loss = (F.cross_entropy(logits, ytr[idx], label_smoothing=label_smoothing)
                    + lambda_reg * F.mse_loss(rpred, rtr[idx]))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()

        agent.eval()
        with torch.no_grad():
            vlogits, vrpred = agent(Xva)
            vloss = (F.cross_entropy(vlogits, yva, label_smoothing=label_smoothing)
                     + lambda_reg * F.mse_loss(vrpred, rva)).item()
            vpred = vlogits.argmax(1).cpu().numpy()
        traded  = (vpred == 1) | (vpred == 2)
        correct = ((vpred == 1) & (fwd_va > 0)) | ((vpred == 2) & (fwd_va < 0))
        vacc = correct[traded].mean() if traded.sum() else float('nan')
        val_acc_hist.append(vacc)

        if vloss < best_loss - 1e-4:
            best_loss, best_ep, bad = vloss, ep, 0
            best_state = {k: v.detach().clone() for k, v in agent.state_dict().items()}
        else:
            bad += 1
        if ep + 1 >= min_epochs and bad >= patience:
            print(f'  Early stopping en epoch {ep+1}')
            break

    if best_state is not None:
        agent.load_state_dict(best_state)
    return val_acc_hist, best_ep

In [ ]:
# --- Preparar features adicionales y construir secuencias para la TCN ---
# Reproducimos la creación de feat_full usada en el notebook TCN original
df_full = processor.data.copy()
df_full['SMA_200'] = df_full['Close'].rolling(200).mean()
feat_full = pd.DataFrame(index=df_full.index)
feat_full['ret_5']     = np.log(df_full['Close'] / df_full['Close'].shift(5))
feat_full['ret_20']    = np.log(df_full['Close'] / df_full['Close'].shift(20))
feat_full['px_sma50']  = df_full['Close'] / df_full['SMA_50']  - 1.0
feat_full['px_sma200'] = df_full['Close'] / df_full['SMA_200'] - 1.0
feat_full['rsi']       = df_full['RSI_14'] / 100.0
feat_full['vol']       = df_full['Volatilidad']
feat_full = feat_full.fillna(0.0)

# Reindexar por train/test y escalar estas features con stats de train (z-score)
feat_train = feat_full.loc[train_df.index]
feat_test  = feat_full.loc[test_df.index]
fmean, fstd = feat_train.mean(), feat_train.std() + 1e-8
feat_train_sc = ((feat_train - fmean) / fstd).values.astype(np.float32)
feat_test_sc  = ((feat_test  - fmean) / fstd).values.astype(np.float32)

# Alinear feats con ventanas (tomamos la fila final de cada ventana como la alineación)
fw_train = feat_train_sc[WINDOW_SIZE - 1: WINDOW_SIZE - 1 + len(X_train)]
fw_test  = feat_test_sc[WINDOW_SIZE - 1: WINDOW_SIZE - 1 + len(X_test)]

# Cargar o usar el modelo entrenado arriba (si prefieres recargar desde disco)
cnn_ae = model  # ya entrenado y en memoria

# Parámetros TCN
TCN_SEQ_LEN = 10
TCN_CHANNELS = [32, 16]
DEVICE = device

# Construir z-sequences (el encoder devuelve flattened latent + feats)
z_train = build_zf_sequences(cnn_ae, X_train, fw_train, TCN_SEQ_LEN, DEVICE)
z_test  = build_zf_sequences(cnn_ae, X_test,  fw_test,  TCN_SEQ_LEN, DEVICE)
print('z_train shape, z_test shape =', z_train.shape, z_test.shape)

# Alinear dates para las decisiones: DEC_OFFSET compensa WINDOW_SIZE y TCN_SEQ_LEN
DEC_OFFSET = WINDOW_SIZE + TCN_SEQ_LEN - 2
test_dec_dates = test_df.index[DEC_OFFSET: DEC_OFFSET + len(z_test)]

# Targets (fwd returns)
close_train = df_full['Close'].loc[train_df.index]
close_test  = df_full['Close'].loc[test_df.index]
fwd_train = forward_returns(close_train, DEC_OFFSET, len(z_train), horizon=8)
fwd_test  = forward_returns(close_test,  DEC_OFFSET, len(z_test),  horizon=8)
r1_train = forward_returns(close_train, DEC_OFFSET, len(z_train), horizon=1)
# Trimear si hace falta
minlen = min(len(z_train), len(fwd_train), len(r1_train))
z_train, fwd_train, r1_train = z_train[:minlen], fwd_train[:minlen], r1_train[:minlen]

# Instanciar agente y entrenar
Z_DIM = z_train.shape[2]
agent = SemaforoAgent(z_dim=Z_DIM, tcn_channels=TCN_CHANNELS, tcn_kernel=3, dropout=0.3, context_dim=64).to(DEVICE)
print('Parámetros entrenables:', sum(p.numel() for p in agent.parameters() if p.requires_grad))

history, best_ep = train_supervised(agent, z_train, fwd_train, r1_train, epochs=100, lr=3e-4, device=DEVICE)
plt.figure(figsize=(8,3)); plt.plot(history); plt.title('val_acc (TCN)'); plt.show()

In [ ]:
# --- Inferencia y semáforo final ---
def run_inference_df(agent, z_sequences, dates, device, coverage=0.30):
    agent.eval()
    P, R = [], []
    for i in range(len(z_sequences)):
        x = torch.tensor(z_sequences[i], dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            logits, rpred = agent(x)
            P.append(F.softmax(logits, dim=-1).squeeze().cpu().numpy())
            R.append(float(rpred.cpu()))
    P, R = np.array(P), np.array(R)
    score = P[:,1] - P[:,2]
    lo = np.quantile(score, coverage/2)
    hi = np.quantile(score, 1 - coverage/2)
    actions = np.where(score >= hi, 1, np.where(score <= lo, 2, 0))
    return pd.DataFrame({
        'date': dates[:len(actions)], 'action': actions,
        'label': [ {0:'HOLD',1:'BUY',2:'SELL'}[a] for a in actions],
        'p_hold': P[:,0], 'p_buy': P[:,1], 'p_sell': P[:,2], 'r_pred': R,
    }).set_index('date')

signals = run_inference_df(agent, z_test, test_dec_dates, DEVICE, coverage=0.30)
print(signals['label'].value_counts())
# Aplicar filtro de tendencia simple (opcional)
def apply_trend_gate_simple(signals, df, fast=50, dd_look=20, dd_thr=-0.08):
    close = df['Close']
    sma = close.rolling(fast).mean()
    ret_r = np.log(close / close.shift(dd_look))
    bullish = ((close > sma) & (ret_r > dd_thr)).reindex(signals.index).fillna(False).values
    act = signals['action'].values.copy()
    act[bullish & (act == 2)] = 0
    act[~bullish & (act == 1)] = 0
    out = signals.copy()
    out['action'] = act
    out['label'] = [ {0:'HOLD',1:'BUY',2:'SELL'}[a] for a in act ]
    return out

signals_f = apply_trend_gate_simple(signals, df_full)
plot_semaforo(signals_f, df_full['Close'].loc[test_df.index], title='Semáforo final (TCN + filtro)')

## Conclusiones y siguientes pasos
- Este notebook reproduce el flujo del repositorio: preprocesado → CNN-AE → MSE-semaforo → TCN-agent → semáforo final.
- Recomendaciones: probar varios `FEATURE_COLS`, ajustar `WINDOW_SIZE`, experimentar con la dimensión latente y `TCN` depth/width, y realizar backtests económicos reales (incluyendo costes de transacción).

¿Quieres que ejecute el notebook localmente aquí (ejecutar celdas y guardar salida), o que adapte alguna celda para correr en Google Colab específicamente?